# L4 35 — train the context-aware Qwen 3B link

Fine-tunes the original link on benign, context-randomized collaboration examples. Sender states are extracted after realistic prefixes, and the mapped representation is injected after the receiver's full instruction—the same structural placement used in the arena.

No game prompts, histories, actions, rewards, or arena results enter training. The frozen base model is never updated. Checkpoints are written to Drive and re-running resumes automatically.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = '8db85f1b23d640f38135b1ac09c4cfb9da394de8'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
SOURCE_JOB_ID = 'faithful-qwen3b-t4-001'
JOB_ID = 'contextual-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
STEPS = 1000
ROOT = pathlib.Path('/content/drive/MyDrive/rival-arena-l4')
SOURCE_LINK = ROOT/SOURCE_JOB_ID/'faithful_link.pt'
JOB_DIR = ROOT/JOB_ID
JOB_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE_LINK.exists(), 'Missing original trained link in Drive'
print('Output:', JOB_DIR)

In [ ]:
command = [
    'python', 'scripts/train_context_link.py',
    '--model', MODEL,
    '--output', JOB_DIR,
    '--job-id', JOB_ID,
    '--init-link', SOURCE_LINK,
    '--steps', str(STEPS),
    '--gradient-checkpointing',
]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected output: `MyDrive/rival-arena-l4/contextual-qwen3b-t4-001/`. Keep the original `faithful-qwen3b-t4-001` directory; this creates a separate adapter lineage and does not overwrite the negative confirmatory result.